<a id="top"></a>

# Lab L0303: build your own single node

```yaml
title: "Lab L0303: build your own single node"
keywords: langgraph, stategraph, node, typed state, typeddict, state update, pure function, compile, invoke, single node, summarize, lab
estimated duration: 25
```

> **Lesson:** L03. See [objectives.md](../../../../docs/origin/lesson_roadmaps/L03/objectives.md).
> **Solutions:** `L0303_lab_solutions.ipynb`.
> Runs **fully offline** -- a deterministic `StubChat` stands in for `ChatAnthropic`, so you can
> focus on *wiring one node*. No API key needed.

## Contents

- [1. Setup](#1-setup)
- [2. Problem 1 -- the typed state](#2-problem-1----the-typed-state)
- [3. Problem 2 -- the node](#3-problem-2----the-node)
- [4. Problem 3 -- wire, compile, render](#4-problem-3----wire-compile-render)
- [5. Problem 4 -- invoke and inspect](#5-problem-4----invoke-and-inspect)
- [6. Problem 5 -- from stub to real model (written)](#6-problem-5----from-stub-to-real-model-written)
- [7. Self-check](#7-self-check)

## 1. Setup

Everything below is **given**: the offline `StubChat`/`StubReply` stand-in for `ChatAnthropic`, two
sample tickets, and one stub client (`sonnet`). You build the one-node graph. Run this cell first.

In [ ]:
from typing import TypedDict

from langgraph.graph import END, StateGraph

SONNET = "claude-sonnet-4-6"  # the name is real; a stub stands in for the client in this lab

# The running example, same support-ticket domain as the lecture.
TICKETS = {
    "billing": "I was charged twice for my subscription this month -- please refund the extra charge.",
    "technical": "The export button throws a 500 error every time I click it on the reports page.",
}


class StubReply:
    """Mimics the one field we read off a ChatAnthropic response: ``.content``."""

    def __init__(self, content: str) -> None:
        self.content = content


class StubChat:
    """Offline stand-in for ``ChatAnthropic`` so this lab is free, fast, and deterministic.

    It mimics the single call the node uses -- ``.invoke(prompt).content`` -> str. Swapping it for
    ``ChatAnthropic(model=SONNET, api_key=require_anthropic_key())`` is a one-line change, and the
    node code never changes (that is Problem 5). The lecture demo uses the real client.
    """

    def __init__(self, model: str) -> None:
        self.model = model

    def invoke(self, prompt: str) -> StubReply:
        # Deterministic canned summary -- the point of this lab is the WIRING, not the model.
        return StubReply("The customer reports an issue and is asking for help resolving it.")


sonnet = StubChat(SONNET)  # ONE client, used by the ONE node

[↑ Back to top](#top)

## 2. Problem 1 -- the typed state

Define the **state** your one node threads through. It needs just two fields: the input `raw_text`
the node reads, and the output `summary` the node will populate. Keep it to those two -- that is the
smallest state that makes the point. (No reducers, no `Annotated` -- those are L04's, for *merging
multiple* nodes' updates. One node needs neither.)

In [ ]:
# TODO: define SummarizeState with TWO fields:
#   raw_text: str   -- the INPUT the node reads
#   summary:  str   -- the OUTPUT the node populates
# The 1-field stub below just keeps the later cells importable -- REPLACE it.
class SummarizeState(TypedDict):
    raw_text: str

[↑ Back to top](#top)

## 3. Problem 2 -- the node

Write `summarize_node`: a plain typed function `state -> update`. It reads `state["raw_text"]`,
calls the model (`sonnet.invoke(...)`), and returns a **state update** -- a dict of *only the field
it changed* (`{"summary": ...}`), **not** the whole state and **not** "the answer". This is the whole
lesson: *state goes in, a state update comes out.*

In [ ]:
def summarize_node(state: "SummarizeState") -> dict[str, object]:
    # TODO: build a prompt asking for a ONE-SENTENCE summary of state["raw_text"],
    #       call sonnet.invoke(prompt), and return a STATE UPDATE -- only the field you changed:
    #           return {"summary": <the model's text>}
    #       Return an *update*, not the whole state and not "the answer".
    raise NotImplementedError("your code here")

[↑ Back to top](#top)

## 4. Problem 3 -- wire, compile, render

Put the node through the smallest possible `StateGraph`: one node, an entry point, and one edge
straight to `END`. `compile()` it to `summarize_app`, then print the diagram with `draw_mermaid()`.
Notice the render needs no model at all -- the control flow is just **data**. (A one-node diagram is
nearly content-free; in L04, with several nodes, the diagram earns its keep.)

In [ ]:
# TODO: build the smallest possible graph with StateGraph(SummarizeState):
#   add_node("summarize", summarize_node); set_entry_point("summarize");
#   add_edge("summarize", END); then compile() into `summarize_app`.
# Finally, print the diagram: print(summarize_app.get_graph().draw_mermaid())
raise NotImplementedError("your code here")

[↑ Back to top](#top)

## 5. Problem 4 -- invoke and inspect

`invoke()` runs the compiled graph on a specific input. Print the returned state and look at both
fields: `raw_text` is **still there unchanged** (the input survived) and `summary` is now
**populated** (the output appeared). `invoke()` hands back the *whole state*, not just the node's
output -- that is "state comes out" made concrete.

In [ ]:
# TODO: invoke summarize_app on TICKETS["billing"] and print the returned state.
#   Note BOTH: result["raw_text"] is still there (input intact) and result["summary"]
#   is now populated (output added). That is "state comes out".
raise NotImplementedError("your code here")

[↑ Back to top](#top)

## 6. Problem 5 -- from stub to real model (written)

The lecture demo used the real `ChatAnthropic` client; this lab used `StubChat`. In a sentence or
two: **which line(s) would you change** to make `summarize_node` call a real model, and **why does
the node function itself not change**? (Hint: what interface do `StubChat` and `ChatAnthropic`
share?)

_Write your answer by editing this cell (double-click)._

_TODO_

[↑ Back to top](#top)

## 7. Self-check

Compare against `L0303_lab_solutions.ipynb`. You're done when:

- `SummarizeState` has exactly two fields (`raw_text`, `summary`), no reducers.
- `summarize_node` returns a **partial update** (`{"summary": ...}`), never the whole state or a bare
  string.
- `summarize_app` compiles and `draw_mermaid()` shows a single node wired straight to `END`.
- `invoke()` returns a state where `raw_text` is intact **and** `summary` is populated.
- You can say, in one sentence, why swapping `StubChat` for `ChatAnthropic` doesn't touch the node
  code (both expose `.invoke(prompt).content`).

> **You just wired one step -- next lesson, you wire several of them together.**

[↑ Back to top](#top)